# 06 — Kiểm chứng Kafka → Spark → HDFS
Dừng service `streaming-consumer` trước khi chạy notebook này; không để hai tiến trình cùng dùng checkpoint. Kafka connector phải được cấu hình trước khi tạo SparkSession. Event sai làm query thất bại. Replay cùng ID vẫn là nhiều sự kiện.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from spark_common import create_spark_session
from project_config import DATA_DIR, RESULTS_DIR, OUTPUT_DIR
from spark_streaming_consumer import start_stream
from kafka_producer import stream_papers_simulation
from project_config import HDFS_ROOT
spark = create_spark_session("ArXivStreamingNotebook", kafka=True)
query = None
try:
    query = start_stream(spark)
    acknowledged = stream_papers_simulation(num_records=10, delay_sec=0.1,
                      bootstrap_servers="kafka:29092")
    query.processAllAvailable()
    print("Acknowledged:", acknowledged)
    print("Spark progress:", query.lastProgress)
    spark.read.parquet(f"{HDFS_ROOT}/data/arxiv_stream_v2").select("id", "year", "main_group_clean").show(20)
finally:
    if query is not None:
        query.stop()
    spark.stop()
